In [2]:
import anthropic
import json
import numpy as np
import pandas as pd

client = anthropic.Anthropic()
def get_car_specs(make, model, fuel_type):
    prompt = f"""For the car {make} {model} with fuel type "{fuel_type}", provide the following in strict JSON format only, no other text:
{{
  "cylinders": <number, integer, or null if electric with no engine>,
  "fuel_efficiency_l_per_100km": <number, float, or null if fully electric>,
  "country_of_origin": "<string>"
}}
Consider that the fuel type may indicate a specific trim/variant (e.g. M-performance, Hybrid, Electric) which can have different specs than the base model.
If unsure about a specific value, use null."""

    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=200,
        messages=[{"role": "user", "content": prompt}]
    )
    
    text = response.content[0].text.strip()
    text = text.replace("```json", "").replace("```", "").strip()
    return json.loads(text)

In [3]:
df= pd.read_csv("PreEDAVersion.csv")

In [ ]:

results = []

for idx, row in unique_cars.iterrows():
    try:
        specs = get_car_specs(row["Make"], row["Model"], row["fuel_type"])
        specs["Make"] = row["Make"]
        specs["model"] = row["Model"]
        specs["fuel_type"] = row["fuel_type"]
        print(f"Success  for {row['Make']} {row['Model']} ({row['fuel_type']})")
        results.append(specs)
    except Exception as e:
        print(f"Failed for {row['Make']} {row['Model']} ({row['fuel_type']}): {e}")
        results.append({"Make": row["Make"], "Model": row["Model"], "fuel_type": row["fuel_type"],
                        "cylinders": None, "fuel_efficiency_l_per_100km": None,
                        "country_of_origin": None})

specs_df = pd.DataFrame(results)
specs_df.to_csv("DataEnrichment.csv")

In [7]:
client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY

def get_body_type(make, model, more_info):
    info_line = f'Additional info: "{more_info}"' if pd.notna(more_info) else "No additional info provided."
    
    prompt = f"""For the car {make} {model}, determine its body type (e.g. Sedan, SUV, Coupe, Hatchback, Pickup Truck, Convertible, Wagon, Minivan).

{info_line}

If the additional info explicitly states or clearly implies the body type, use that. Otherwise, use your own knowledge of this make and model to determine the most likely body type.

Respond in strict JSON format only, no other text:
{{
  "body_type": "<string>"
}}"""

    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=100,
        messages=[{"role": "user", "content": prompt}]
    )
    
    text = response.content[0].text.strip()
    text = text.replace("```json", "").replace("```", "").strip()
    return json.loads(text)

In [8]:
unique_cars = df[["Make", "Model", "MoreInformation"]].drop_duplicates().reset_index(drop=True)
unique_cars

,Make,Model,MoreInformation
0,Lincoln,Corsair,NaN
1,Jeep,Cherokee,NaN
2,Hyundai,Tucson,NaN
3,Mazda,CX-5,NaN
4,Ford,F-150,NaN
...,...,...,...
778,Porsche,911,Carrera 4 GTS Coupe
779,Nissan,GT-R,NaN
780,BMW,5,Series Sedan
781,Audi,TT,RS


In [9]:
unique_cars = df[["Make", "Model", "MoreInformation"]].drop_duplicates().reset_index(drop=True)

results = []

for idx, row in unique_cars.iterrows():
    try:
        specs = get_body_type(row["Make"], row["Model"], row["MoreInformation"])
        specs["Make"] = row["Make"]
        specs["Model"] = row["Model"]
        specs["MoreInformation"] = row["MoreInformation"]
        print(f"Success for {row['Make']} {row['Model']}")
        results.append(specs)
    except Exception as e:
        print(f"Failed for {row['Make']} {row['Model']}: {e}")
        results.append({"Make": row["Make"], "Model": row["Model"], 
                        "MoreInformation": row["MoreInformation"], "body_type": None})

body_type_df = pd.DataFrame(results)

Success for Lincoln Corsair
Succceful for Lincoln Corsair 
Success for Jeep Cherokee
Succceful for Jeep Cherokee 
Success for Hyundai Tucson
Succceful for Hyundai Tucson 
Success for Mazda CX-5
Succceful for Mazda CX-5 
Success for Ford F-150
Succceful for Ford F-150 
Success for Land Rover
Succceful for Land Rover 
Success for Honda Passport
Succceful for Honda Passport 
Success for Kia Seltos
Succceful for Kia Seltos 
Success for Genesis GV70
Succceful for Genesis GV70 
Success for Audi S5
Succceful for Audi S5 
Success for Mazda Mazda3
Succceful for Mazda Mazda3 
Success for Subaru Outback
Succceful for Subaru Outback 
Success for Toyota Prius
Succceful for Toyota Prius 
Success for BMW X5
Succceful for BMW X5 
Success for Subaru WRX
Succceful for Subaru WRX 
Success for Infiniti QX60
Succceful for Infiniti QX60 
Success for Ford Bronco
Succceful for Ford Bronco 
Success for Honda Civic
Succceful for Honda Civic 
Success for Nissan Versa
Succceful for Nissan Versa 
Success for Merce

In [11]:
body_type_df.to_csv("BodyType.csv")